In [18]:
import polars as pl

rna_seq_sources = ['Hugo et al.', 'Kwong et al.', 'Yan et al.']
q_pcr_sources = ['Louveau et al.']
micro_array_sources = ['Long et al.', 'Rizos et al.']

source_map = {
    'doi:10.1016/j.cell.2015.07.061': 'Hugo et al.',
    'doi:10.1172/JCI78954DS1': 'Kwong et al.',
    'doi:10.1158/1078-0432.CCR-18-0720': 'Yan et al.',
    'doi:10.3390/cancers11081203': 'Louveau et al.',
    'doi:10.1038/ncomms6694': 'Long et al.',
    'doi:10.1158/1078-0432.CCR-13-3122': 'Rizos et al.'
}

gex = pl.read_csv("../dataset/original/gene_expressions.csv")
gex = gex.with_columns(pl.col('source').replace(source_map))

In [19]:
DROP_NULLS = False
SELECT_PRE_TREATMENT = True
SELECT_RNA_SEQ = False

DATASET = 'gex_pre'

In [20]:
if SELECT_PRE_TREATMENT == True:
    gex = gex.filter((pl.col('temporality') == 'pre treatment'))
if SELECT_RNA_SEQ == True:
    gex = gex.filter(pl.col('source').is_in(rna_seq_sources))

In [21]:
# Drop useless features
gex = gex.drop(['id', 'creation_datetime', 'GeneID', 'description', 'temporality'])
gex.select(pl.all().null_count())

patientID,sample_id,HGNC,value,source
u32,u32,u32,u32,u32
92181,0,207000,0,0


In [22]:
if DROP_NULLS == True:
    # Sample is useless if patientID or HGNC is not given
    gex = gex.drop_nulls(subset=['patientID', 'HGNC'])
    gex.select(pl.all().null_count())

In [23]:
gex = gex.with_columns(
    pl.when(pl.col('source').is_in(rna_seq_sources))
    .then(pl.lit('RNA-seq'))
    .when(pl.col('source').is_in(micro_array_sources))
    .then(pl.lit('micro-array'))
    .otherwise(pl.lit('qPCR'))
    .alias('Method')
)
gex

patientID,sample_id,HGNC,value,source,Method
str,str,str,f64,str,str
"""LM_1""","""LMSAM_1""","""BRAF""",7.60798,"""Louveau et al.""","""qPCR"""
"""LM_1""","""LMSAM_1""","""RAF1""",16.095204,"""Louveau et al.""","""qPCR"""
"""LM_1""","""LMSAM_1""","""ARAF""",4.1515,"""Louveau et al.""","""qPCR"""
"""LM_1""","""LMSAM_1""","""PDGFRB""",1.199885,"""Louveau et al.""","""qPCR"""
"""LM_1""","""LMSAM_1""","""IGF1R""",5.47246,"""Louveau et al.""","""qPCR"""
…,…,…,…,…,…
"""HL_Shi-43""","""Pt17-baseline""","""ZYG11A""",0.0625681,"""Hugo et al.""","""RNA-seq"""
"""HL_Shi-43""","""Pt17-baseline""","""ZYG11B""",5.74608,"""Hugo et al.""","""RNA-seq"""
"""HL_Shi-43""","""Pt17-baseline""","""ZYX""",45.907933,"""Hugo et al.""","""RNA-seq"""


In [24]:
# gex.write_csv(f'../dataset/created/gex/{DATASET}.csv')

In [25]:
# Count unique patientIDs for each source
unique_patients_per_source = (
    gex.group_by("source")
    .agg(pl.col("patientID").n_unique().alias("unique_patient_count"))
    .sort("unique_patient_count", descending=True)
)

print(unique_patients_per_source)

shape: (6, 2)
┌────────────────┬──────────────────────┐
│ source         ┆ unique_patient_count │
│ ---            ┆ ---                  │
│ str            ┆ u32                  │
╞════════════════╪══════════════════════╡
│ Yan et al.     ┆ 70                   │
│ Rizos et al.   ┆ 21                   │
│ Louveau et al. ┆ 20                   │
│ Kwong et al.   ┆ 14                   │
│ Hugo et al.    ┆ 12                   │
│ Long et al.    ┆ 10                   │
└────────────────┴──────────────────────┘


In [27]:
# Group by source and collect unique patient IDs into a list
patient_ids_by_source = (
    gex.group_by("source")
    .agg(pl.col("patientID").unique().alias("ids"))
    .to_dicts()
)

for entry in patient_ids_by_source:
    print(f"--- Source: {entry['source']} ---")
    print(f"IDs: {entry['ids']}\n")

--- Source: Louveau et al. ---
IDs: ['LM_1', 'LM_2', 'LM_4', 'LM_6', 'LM_7', 'LM_8', 'LM_9', 'LM_10', 'LM_11', 'LM_14', 'LM_15', 'LM_16', 'LM_18', 'LM_19', 'LM_20', 'LM_21', 'LM_23', 'LM_25', 'LM_26', 'LM_27']

--- Source: Yan et al. ---
IDs: [None, 'YR_5306', 'YR_5311', 'YR_nan', 'YR_8902', 'YR_3053', 'YR_3708', 'YR_3705', 'YR_1106', 'YR_1110', 'YR_1402', 'YR_1154', 'YR_2097', 'YR_2078', 'YR_2203', 'YR_2096', 'YR_2208', 'YR_2362', 'YR_2066', 'YR_2083', 'YR_2347', 'YR_2420', 'YR_2094', 'YR_2346', 'YR_2210', 'YR_2281', 'YR_2319', 'YR_2296', 'YR_2261', 'YR_2035', 'YR_2214', 'YR_2220', 'YR_2382', 'YR_2148', 'YR_2191', 'YR_2207', 'YR_2174', 'YR_2378', 'YR_2136', 'YR_2484', 'YR_2048', 'YR_2472', 'YR_2459', 'YR_2337', 'YR_2434', 'YR_2063', 'YR_2131', 'YR_2211', 'YR_2300', 'YR_2443', 'YR_2390', 'YR_2036', 'YR_2363', 'YR_2425', 'YR_2405', 'YR_2380', 'YR_2466', 'YR_2409', 'YR_2496', 'YR_2494', 'YR_2417', 'YR_112008', 'YR_102011', 'YR_113005', 'YR_106006', 'YR_106008', 'YR_103001', 'YR_104002', 

In [28]:
# 1. Define the Clinical IDs based on your provided list
clinical_data = {
    'Louveau et al.': ['LM_1', 'LM_10', 'LM_11', 'LM_13', 'LM_14', 'LM_15', 'LM_16', 'LM_17', 'LM_18', 'LM_19', 'LM_2', 'LM_20', 'LM_21', 'LM_22', 'LM_23', 'LM_25', 'LM_26', 'LM_27', 'LM_3', 'LM_4', 'LM_6', 'LM_7', 'LM_8', 'LM_9'],
    'Long et al.': ['LR_MTP-034', 'LR_MTP-073', 'LR_SMU-020', 'LR_SMU-028', 'LR_SMU-030', 'LR_SMU-034', 'LR_WMD-017', 'LR_WMD-022', 'LR_WMD-033'],
    'Rizos et al.': ['RL_MTP-001', 'RL_MTP-014', 'RL_MTP-020', 'RL_MTP-063', 'RL_MTP-064', 'RL_MTP-066', 'RL_MTP-080', 'RL_MTP-085', 'RL_MTP-093', 'RL_MTP-095', 'RL_MTP-099', 'RL_SMU-017', 'RL_WMD-006', 'RL_WMD-007', 'RL_WMD-009', 'RL_WMD-012', 'RL_WMD-013', 'RL_WMD-021', 'RL_WMD-025', 'RL_WMD-027', 'RL_WMD-028', 'RL_WMD-029', 'RL_WMD-035', 'RL_WMD-039'],
    'Hugo et al.': ['HL_1', 'HL_10', 'HL_11', 'HL_12', 'HL_13', 'HL_14', 'HL_15', 'HL_16', 'HL_17', 'HL_18', 'HL_19', 'HL_2', 'HL_20', 'HL_21', 'HL_22', 'HL_23', 'HL_24', 'HL_25', 'HL_26', 'HL_27', 'HL_28', 'HL_29', 'HL_3', 'HL_30', 'HL_31', 'HL_32', 'HL_33', 'HL_34', 'HL_35', 'HL_36', 'HL_37', 'HL_38', 'HL_39', 'HL_4', 'HL_40', 'HL_41', 'HL_42', 'HL_43', 'HL_44', 'HL_5', 'HL_6', 'HL_7', 'HL_8', 'HL_9'],
    'Yan et al.': ['YR_1002', 'YR_1005', 'YR_1006', 'YR_1016', 'YR_102004', 'YR_102005', 'YR_102011', 'YR_1021', 'YR_1026', 'YR_103001', 'YR_103003', 'YR_104002', 'YR_1054', 'YR_1058', 'YR_1060', 'YR_106006', 'YR_106008', 'YR_1067', 'YR_1070', 'YR_1071', 'YR_108004', 'YR_109003', 'YR_110007', 'YR_1106', 'YR_1110', 'YR_1111', 'YR_112008', 'YR_112010', 'YR_112016', 'YR_113005', 'YR_114005', 'YR_114010', 'YR_1151', 'YR_1154', 'YR_1206', 'YR_1256', 'YR_1257', 'YR_1267', 'YR_1302', 'YR_1304', 'YR_1402', 'YR_1451', 'YR_1605', 'YR_1607', 'YR_1760', 'YR_1951', 'YR_2005', 'YR_2035', 'YR_2036', 'YR_2047', 'YR_2048', 'YR_2063', 'YR_2066', 'YR_2078', 'YR_2083', 'YR_2094', 'YR_2096', 'YR_2097', 'YR_2102', 'YR_2112', 'YR_2123', 'YR_2131', 'YR_2136', 'YR_2147', 'YR_2148', 'YR_2159', 'YR_2160', 'YR_2174', 'YR_2191', 'YR_2194', 'YR_2203', 'YR_2207', 'YR_2208', 'YR_2210', 'YR_2211', 'YR_2213', 'YR_2214', 'YR_2219', 'YR_2220', 'YR_2261', 'YR_2281', 'YR_2295', 'YR_2296', 'YR_2300', 'YR_2319', 'YR_2323', 'YR_2337', 'YR_2346', 'YR_2347', 'YR_2362', 'YR_2363', 'YR_2366', 'YR_2377', 'YR_2378', 'YR_2380', 'YR_2382', 'YR_2390', 'YR_2405', 'YR_2409', 'YR_2417', 'YR_2420', 'YR_2425', 'YR_2434', 'YR_2443', 'YR_2459', 'YR_2466', 'YR_2468', 'YR_2471', 'YR_2472', 'YR_2475', 'YR_2484', 'YR_2492', 'YR_2494', 'YR_2496', 'YR_2497', 'YR_3053', 'YR_3151', 'YR_3253', 'YR_3705', 'YR_3708', 'YR_3905', 'YR_4210', 'YR_5306', 'YR_5311', 'YR_7801', 'YR_8902', 'YR_9304', 'YR_9306', 'YR_9403', 'YR_9602'],
    'Kwong et al.': ['KC_10', 'KC_12', 'KC_13', 'KC_15', 'KC_16', 'KC_17', 'KC_19', 'KC_2', 'KC_20', 'KC_21', 'KC_22', 'KC_24', 'KC_25', 'KC_26', 'KC_27', 'KC_28', 'KC_29', 'KC_34', 'KC_4', 'KC_6', 'KC_7', 'KC_9']
}

# 2. Get unique IDs from your current Polars DataFrame
gex_ids_by_source = (
    gex.group_by("source")
    .agg(pl.col("patientID").unique())
    .to_dicts()
)

# 3. Compare and Print Overlaps
print(f"{'Source':<20} | {'GEX IDs':<8} | {'Clin IDs':<8} | {'Overlap'}")
print("-" * 55)

for entry in gex_ids_by_source:
    source_name = entry['source']
    gex_set = set(entry['patientID'])
    clin_set = set(clinical_data.get(source_name, []))
    
    overlap = gex_set.intersection(clin_set)
    
    print(f"{source_name:<20} | {len(gex_set):<8} | {len(clin_set):<8} | {len(overlap)}")

Source               | GEX IDs  | Clin IDs | Overlap
-------------------------------------------------------
Long et al.          | 10       | 9        | 9
Louveau et al.       | 20       | 24       | 20
Hugo et al.          | 12       | 44       | 0
Yan et al.           | 70       | 130      | 68
Kwong et al.         | 14       | 22       | 14
Rizos et al.         | 21       | 24       | 21
